In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import numpy as np
from catboost import CatBoostClassifier
import shap

print("step 1: loading data")
X, y = shap.datasets.california(n_points=200)
y = np.asarray(y)
y_multi = (y > np.median(y)).astype(int) + (y > np.quantile(y, 0.75)).astype(int)

print("step 2: training model")
model = CatBoostClassifier(loss_function="MultiClass", iterations=100, learning_rate=0.1, random_seed=123, verbose=False)
model.fit(X, y_multi)

print("step 3: creating explainer")
explainer = shap.TreeExplainer(model)

print("step 4: computing shap values")
shap_values = explainer(X, y_multi)

print("step 5: done, shape =", shap_values.shape)

step 1: loading data
step 2: training model
step 3: creating explainer
step 4: computing shap values
step 5: done, shape = (200, 8, 3)


In [ ]:
import sys, os, time, json
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
import shap

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix

LABEL_MAP = {'GALAXY': 0, 'QSO': 1, 'STAR': 2}
REVERSE_MAP = {0: 'GALAXY', 1: 'QSO', 2: 'STAR'}

train_raw = pd.read_csv('../data/raw/train.csv')
test_raw  = pd.read_csv('../data/raw/test.csv')

print(f"train_raw: {train_raw.shape}")
print(f"test_raw:  {test_raw.shape}")
print("\nClass distribution (training):")
print(train_raw['class'].value_counts().to_string())


In [ ]:
BAND_COLS = ['u', 'g', 'r', 'i', 'z']
SPECTRAL_TYPES = ['A/F', 'G/K', 'M', 'O/B']
GALAXY_POPULATIONS = ['Blue_Cloud', 'Red_Sequence']

def build_features(df):
    df = df.copy()
    df['u_g'] = df['u'] - df['g']
    df['g_r'] = df['g'] - df['r']
    df['r_i'] = df['r'] - df['i']
    df['i_z'] = df['i'] - df['z']
    df['g_i'] = df['g'] - df['i']
    df['r_z'] = df['r'] - df['z']
    df['u_r'] = df['u'] - df['r']
    df['g_z'] = df['g'] - df['z']
    df['u_z'] = df['u'] - df['z']
    df['u_i'] = df['u'] - df['i']

    df['band_mean'] = df[BAND_COLS].mean(axis=1)
    df['band_std'] = df[BAND_COLS].std(axis=1)
    df['band_range'] = df[BAND_COLS].max(axis=1) - df[BAND_COLS].min(axis=1)
    df['band_median'] = df[BAND_COLS].median(axis=1)

    df['redshift_log1p'] = np.log1p(df['redshift'].clip(lower=-0.999999))
    df['redshift_sq'] = df['redshift'] ** 2
    df['redshift_abs'] = df['redshift'].abs()
    df['is_high_z'] = (df['redshift'] > 1.0).astype(int)
    df['is_star_z'] = (df['redshift'].abs() < 0.01).astype(int)
    df['is_negative_z'] = (df['redshift'] < 0).astype(int)

    df['delta_abs'] = df['delta'].abs()

    df['spectral_type_cat'] = pd.Categorical(df['spectral_type'], categories=SPECTRAL_TYPES).codes
    df['galaxy_population_cat'] = pd.Categorical(df['galaxy_population'], categories=GALAXY_POPULATIONS).codes

    df['g_r_x_redshift'] = df['g_r'] * df['redshift']
    df['u_g_x_redshift'] = df['u_g'] * df['redshift']
    return df

FEATURE_COLS = [
    'alpha', 'delta', 'delta_abs', 'u', 'g', 'r', 'i', 'z',
    'u_g', 'g_r', 'r_i', 'i_z', 'g_i', 'r_z', 'u_r', 'g_z', 'u_z', 'u_i',
    'band_mean', 'band_std', 'band_range', 'band_median',
    'redshift', 'redshift_log1p', 'redshift_sq', 'redshift_abs',
    'is_high_z', 'is_star_z', 'is_negative_z',
    'spectral_type_cat', 'galaxy_population_cat',
    'g_r_x_redshift', 'u_g_x_redshift',
]

train_fe = build_features(train_raw)
test_fe = build_features(test_raw)

X = train_fe[FEATURE_COLS].reset_index(drop=True)
y = train_fe['class'].map(LABEL_MAP).reset_index(drop=True)
X_test = test_fe[FEATURE_COLS].reset_index(drop=True)

print(f"X:      {X.shape}")
print(f"X_test: {X_test.shape}")
print(f"Null check: {X.isnull().sum().sum()} nulls in X")


In [ ]:
N_SPLITS = 3
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

n_train, n_test = len(X), len(X_test)
oof_lgb = np.zeros((n_train, 3)); test_lgb = np.zeros((n_test, 3))
oof_xgb = np.zeros((n_train, 3)); test_xgb = np.zeros((n_test, 3))
oof_cat = np.zeros((n_train, 3)); test_cat = np.zeros((n_test, 3))
lgb_models, xgb_models, cat_models = [], [], []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"===== FOLD {fold+1}/{N_SPLITS} =====  train={len(tr_idx):,}  val={len(val_idx):,}")
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    t0 = time.time()
    m_lgb = lgb.LGBMClassifier(
        n_estimators=500, learning_rate=0.06, num_leaves=80, max_depth=8,
        min_child_samples=25, subsample=0.85, colsample_bytree=0.8,
        reg_alpha=0.2, reg_lambda=0.3, objective='multiclass', num_class=3,
        class_weight='balanced', n_jobs=1, verbosity=-1, random_state=42,
    )
    m_lgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(40, verbose=False)])
    oof_lgb[val_idx] = m_lgb.predict_proba(X_val)
    test_lgb += m_lgb.predict_proba(X_test) / N_SPLITS
    lgb_models.append(m_lgb)
    ba = balanced_accuracy_score(y_val, oof_lgb[val_idx].argmax(1))
    print(f"  LightGBM fit={time.time()-t0:.1f}s | best_iter={m_lgb.best_iteration_} | fold BA={ba:.4f}")

    t0 = time.time()
    m_xgb = xgb.XGBClassifier(
        n_estimators=350, learning_rate=0.08, max_depth=7,
        subsample=0.85, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.5,
        objective='multi:softprob', num_class=3, n_jobs=1, tree_method='hist',
        early_stopping_rounds=30, eval_metric='mlogloss', verbosity=0, random_state=42,
    )
    m_xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    oof_xgb[val_idx] = m_xgb.predict_proba(X_val)
    test_xgb += m_xgb.predict_proba(X_test) / N_SPLITS
    xgb_models.append(m_xgb)
    ba = balanced_accuracy_score(y_val, oof_xgb[val_idx].argmax(1))
    print(f"  XGBoost  fit={time.time()-t0:.1f}s  | best_iter={m_xgb.best_iteration} | fold BA={ba:.4f}")

    t0 = time.time()
    m_cat = CatBoostClassifier(
        iterations=350, learning_rate=0.08, depth=7,
        loss_function='MultiClass', eval_metric='TotalF1',
        early_stopping_rounds=30, auto_class_weights='Balanced',
        random_seed=42, verbose=0, thread_count=1,
    )
    m_cat.fit(X_tr, y_tr, eval_set=(X_val, y_val))
    oof_cat[val_idx] = m_cat.predict_proba(X_val)
    test_cat += m_cat.predict_proba(X_test) / N_SPLITS
    cat_models.append(m_cat)
    ba = balanced_accuracy_score(y_val, oof_cat[val_idx].argmax(1))
    print(f"  CatBoost fit={time.time()-t0:.1f}s  | best_iter={m_cat.get_best_iteration()} | fold BA={ba:.4f}")
    print()


## OOF Results by Model

In [ ]:
ba_lgb_full = balanced_accuracy_score(y, oof_lgb.argmax(1))
ba_xgb_full = balanced_accuracy_score(y, oof_xgb.argmax(1))
ba_cat_full = balanced_accuracy_score(y, oof_cat.argmax(1))
print(f"LGBM OOF BA:     {ba_lgb_full:.5f}")
print(f"XGBoost OOF BA:  {ba_xgb_full:.5f}")
print(f"CatBoost OOF BA: {ba_cat_full:.5f}")


In [ ]:
best_score, best_w = -np.inf, (1/3, 1/3, 1/3)
for w_lgb in np.arange(0.0, 1.01, 0.05):
    for w_xgb in np.arange(0.0, 1.01 - w_lgb, 0.05):
        w_cat = 1.0 - w_lgb - w_xgb
        if w_cat < -1e-9:
            continue
        w_cat = max(w_cat, 0.0)
        blend = oof_lgb * w_lgb + oof_xgb * w_xgb + oof_cat * w_cat
        score = balanced_accuracy_score(y, blend.argmax(1))
        if score > best_score:
            best_score = score
            best_w = (round(w_lgb, 2), round(w_xgb, 2), round(w_cat, 2))

print(f"Best weights (LGB, XGB, CAT) = {best_w}")
print(f"Best OOF Balanced Accuracy   = {best_score:.5f}")

w_lgb, w_xgb, w_cat = best_w
oof_blend = oof_lgb * w_lgb + oof_xgb * w_xgb + oof_cat * w_cat
test_blend = test_lgb * w_lgb + test_xgb * w_xgb + test_cat * w_cat


## Classification Report & Confusion Matrix (OOF, blended)

In [ ]:
print(classification_report(y, oof_blend.argmax(1), target_names=['GALAXY', 'QSO', 'STAR']))
cm = confusion_matrix(y, oof_blend.argmax(1))
print("Confusion matrix (rows=true, cols=pred) [GALAXY, QSO, STAR]:")
print(cm)


## Feature Importance

In [ ]:
importances = np.zeros(len(FEATURE_COLS))
for m in lgb_models:
    importances += m.feature_importances_ / len(lgb_models)

fi = pd.Series(importances, index=FEATURE_COLS).sort_values(ascending=True)
plt.figure(figsize=(9, 8))
fi.plot(kind='barh', color='#4C72B0')
plt.title('LightGBM Feature Importance (avg over 3 folds)', fontweight='bold')
plt.xlabel('Importance (gain-based split count)')
plt.tight_layout()
plt.savefig('../outputs/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
explainer = shap.TreeExplainer(cat_models[0])
display_sample = X.sample(n=150, random_state=42)
y_display = y.loc[display_sample.index]

shap_values = explainer(display_sample, y_display)
print(f"SHAP values shape: {shap_values.shape}")

STAR_CLASS = 2
plt.figure(figsize=(11, 7))
shap.plots.beeswarm(shap_values[..., STAR_CLASS], max_display=12, show=False)
plt.title('SHAP Summary — STAR class (CatBoost)', fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/shap_beeswarm_star.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
mean_abs = np.abs(shap_values).mean(axis=(0, 2))
order = np.argsort(mean_abs)
plt.figure(figsize=(9, 6))
plt.barh(np.array(FEATURE_COLS)[order][-12:], mean_abs[order][-12:], color='#DD8452')
plt.title('Mean |SHAP value| (avg across classes)', fontweight='bold')
plt.xlabel('Mean |SHAP value|')
plt.tight_layout()
plt.savefig('../outputs/shap_importance_bar.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
test_preds = np.array([REVERSE_MAP[i] for i in test_blend.argmax(1)])
submission = pd.DataFrame({'id': test_raw['id'], 'class': test_preds})
submission.to_csv('../data/processed/submission2.csv', index=False)

dist = submission['class'].value_counts().to_dict()
print("Submission saved: ../data/processed/submission2.csv")
print(f"Class distribution: {dist}")
submission.head()
